# IMPLEMENT SKIP-GRAM MODEL USING NEGATIVE SAMPLING

## STEP 1 - Reading the dataset

The dataset that we use here is Penn Tree Bank (PTB). This corpus is sampled from Wall Street Journal articles, split into training, validation, and test sets. 

In the original format, each line of the text file represents a sentence of words that are separated by spaces. Here we treat each word as a token.

In [1]:
import os
import torch
from utils import spy

def read_ptb():
    """Load the PTB dataset into a list of text lines"""
    data_dir = spy.download_extract('http://d2l-data.s3-accelerate.amazonaws.com/ptb.zip', '../../data')
    # Read the training set
    with open(os.path.join(data_dir, 'ptb.train.txt')) as f:
        raw_text = f.read()
    return [line.split() for line in raw_text.split('\n')]

sentences = read_ptb()
print(f"# sentences: {len(sentences)}")

# sentences: 42069


After reading the training set, we build a vocabulary for the corpus, where any word that appears less than 10 times is replaced by the `<unk>` token. 

Note that the original dataset also contains `<unk>` tokens that represent rare (unknown) words.

In [2]:
vocab = spy.Vocab(sentences, min_freq=10)
f'Vocab size: {len(vocab)}'

'Vocab size: 6719'

## STEP 2 - Subsampling

Text data typically have high-frequency words such as “the”, “a”, and “in”: they may even occur billions of times in very large corpora. 
However, these words often co-occur with many different words in context windows, providing little useful signals. 

For instance, consider the word “chip” in a context window: intuitively its co-occurrence with a low-frequency word “intel” is more useful in training than the co-occurrence with a high-frequency word “a”. Moreover, training with vast amounts of (high-frequency) words is slow. Thus, when training word embedding models, high-frequency words can be subsampled (Mikolov et al., 2013). Specifically, each indexed word $w_i$ in the dataset will be discarded with probability:

$P(w_i) = \max\left( 1 - \sqrt{\frac{t}{f(w_i)}},\, 0 \right)$


In [3]:
subsampled, counter = spy.subsample(sentences, vocab)

Next, we define a function to measure the difference of token counts before and after subsampling.

In [4]:
def compare_counts(token):
    return (f"# of '{token}': before = {sum(l.count(token) for l in sentences)}, " 
                            f"after = {sum(l.count(token) for l in subsampled)}")

print(compare_counts("the"))
print(compare_counts("computer"))

# of 'the': before = 50770, after = 2003
# of 'computer': before = 420, after = 182


After subsampling, we map tokens to their indices for the corpus.

In [5]:
corpus = [vocab[line] for line in subsampled]
corpus[:4]

[[],
 [3228],
 [4103, 3922, 1922, 4743],
 [4127, 289, 4103, 1325, 2641, 2340, 4465, 799, 3039, 1291]]

## STEP 3 - Extracting center words and context words

The following `get_centers_and_contexts` function extracts all the center words and their context words from corpus. 

It uniformly samples an integer between 1 and `max_window_size` at random as the context window size. For any center word, those words whose distance from it does not exceed the sampled context window size are its context words.

Next, we create an artificial dataset containing two sentences of 7 and 3 words, respectively. Let the maximum context window size be 2 and print all the center words and their context words.



In [6]:
tiny_dataset = [list(range(7)), list(range(7, 10))]
print('dataset', tiny_dataset)
for center, context in zip(*spy.get_centers_and_contexts(tiny_dataset, 2)):
    print('center', center, 'has context', context)

dataset [[0, 1, 2, 3, 4, 5, 6], [7, 8, 9]]
center 0 has context [1, 2]
center 1 has context [0, 2]
center 2 has context [1, 3]
center 3 has context [2, 4]
center 4 has context [2, 3, 5, 6]
center 5 has context [3, 4, 6]
center 6 has context [5]
center 7 has context [8, 9]
center 8 has context [7, 9]
center 9 has context [7, 8]


When training on the PTB dataset, we set the maximum context window size to 5. The following extracts all the center words and their context words in the dataset.



In [7]:
all_centers, all_contexts = spy.get_centers_and_contexts(corpus, 5)
f'# center-context pairs: {sum([len(contexts) for contexts in all_contexts])}'

'# center-context pairs: 1501466'

## STEP 4 - Negative Sampling

We use negative sampling for approximate training. To sample noise words according to a predefined distribution, we define the following `RandomGenerator` class, where the (possibly unnormalized) sampling distribution is passed via the argument `sampling_weights`.

For example, we can draw 10 random variables $X$ among indices 1, 2 and 3 with sampling probabilites $P(X = 1) = 2/9, P(X = 2) = 3/9, P(X = 3) = 4/9$ as follows:

In [8]:
generator = spy.RandomGenerator([2, 3, 4])
[generator.draw() for _ in range(10)]

[2, 2, 3, 3, 1, 1, 1, 2, 3, 2]

For a pair of center word and context word, we randomly sample `K` (5 in the experiment) noise words. According to the suggestions in the word2vec paper, the sampling probability $P(w)$ of a noise word $w$ is set to its relative frequency in the dictionary raised to the power of 0.75:

$P(w_i) = \frac{f(w_i)^{0.75}}{\sum_{j \in \mathcal{V}} f(w_j)^{0.75}}$

In [9]:
all_negatives = spy.get_negatives(all_contexts, vocab, counter, 5)

## STEP 5 - Loading training examples in minibatches

After all the center words together with their context words and sampled noise words are extracted, they will be transformed into minibatches of examples that can be iteratively loaded during training.

In a minibatch, the $i^{th}$ example includes a center word and its $n_i$ context words and $m_i$ noise words. Due to varying context window sizes, 
$n_i + m_i$ varies for different $i$. Thus, for each example we concatenate its context words and noise words in the `contexts_negatives` variable, and pad zeros until the concatenation length reaches $max_i n_i + m_i$ (`max_len`). To exclude paddings in the calculation of the loss, we define a mask variable `masks`. There is a one-to-one correspondence between elements in `masks` and elements in `contexts_negatives`, where zeros (otherwise ones) in `masks` correspond to paddings in `contexts_negatives`.

To distinguish between positive and negative examples, we separate context words from noise words in `contexts_negatives` via a `labels` variable. Similar to `masks`, there is also a one-to-one correspondence between elements in `labels` and elements in `contexts_negatives`, where ones (otherwise zeros) in `labels` correspond to context words (positive examples) in `contexts_negatives`.

The above idea is implemented in the following `batchify` function. Its input data is a list with length equal to the batch size, where each element is an example consisting of the center word center, its context words `context`, and its noise words `negative`. This function returns a minibatch that can be loaded for calculations during training, such as including the mask variable.

In [12]:
def batchify(data):
    """Return a minibatch of examples for skip-gram with negative sampling"""
    max_len = max(len(c) + len(n) for _, c, n in data)
    centers, contexts_negatives, masks, labels = [], [], [], []
    
    for center, context, negative in data:
        cur_len = len(context) + len(negative)
        centers += [center]
        contexts_negatives += [context + negative + [0] * (max_len - cur_len)]
        masks += [[1] * cur_len + [0] * (max_len - cur_len)]
        labels += [[1] * len(context) + [0] * (max_len - len(context))]
    
    return (torch.tensor(centers).reshape((-1, 1)), torch.tensor(contexts_negatives), 
            torch.tensor(masks), torch.tensor(labels))

Let’s test this function using a minibatch of two examples.

In [13]:
x_1 = (1, [2, 2], [3, 3, 3, 3])
x_2 = (1, [2, 2, 2], [3, 3])
batch = batchify((x_1, x_2))

names = ['centers', 'contexts_negatives', 'masks', 'labels']
for name, data in zip(names, batch):
    print(name, '=', data)

centers = tensor([[1],
        [1]])
contexts_negatives = tensor([[2, 2, 3, 3, 3, 3],
        [2, 2, 2, 3, 3, 0]])
masks = tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])
labels = tensor([[1, 1, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0]])


## STEP 6 - Putting it all together

Last, we define the `load_data_ptb` function that reads the PTB dataset and returns the data iterator and the vocabulary.

In [16]:
def load_data_ptb(batch_size, max_window_size, num_noise_words):
    """Download the PTB dataset and then load it into memory"""
    sentences = read_ptb()
    vocab = spy.Vocab(sentences, min_freq=10)
    subsampled, counter = spy.subsample(sentences, vocab)
    corpus = [vocab[line] for line in subsampled]
    all_centers, all_contexts = spy.get_centers_and_contexts(corpus, max_window_size)
    all_negatives = spy.get_negatives(all_contexts, vocab, counter, num_noise_words)

    
    class PTBDataset(torch.utils.data.Dataset):
        def __init__(self, centers, contexts, negatives):
            assert len(centers) == len(contexts) == len(negatives)
            self.centers = centers
            self.contexts = contexts
            self.negatives = negatives


        def __getitem__(self, index):        # Allow an object to behave like a List or a Dictionary.
            return (self.centers[index], self.contexts[index], self.negatives[index])


        def __len__(self):
            return len(self.centers)
        
    
    dataset = PTBDataset(all_centers, all_contexts, all_negatives)
    # collate_fn: a custom function that can be passed to the DataLoader to teach it how to package
    # individual samples into a batch.
    data_iter = torch.utils.data.DataLoader(dataset, batch_size, shuffle=True, collate_fn=batchify)

    return data_iter, vocab


Let’s print the first minibatch of the data iterator.

In [17]:
data_iter, vocab = load_data_ptb(512, 5, 5)
names = ['centers', 'context_negatives', 'masks', 'labels']

for batch in data_iter:
    for name, data in zip(names, batch):
        print(name, 'shape:', data.shape)
    break

centers shape: torch.Size([512, 1])
context_negatives shape: torch.Size([512, 60])
masks shape: torch.Size([512, 60])
labels shape: torch.Size([512, 60])
